In [1]:
import torch
import torch.nn.functional as F

from torch_geometric.nn import GATConv
from datasets import CVFGATGeometricDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn.pool import global_mean_pool

In [2]:
device = "cuda"
batch_size = 16

In [3]:
dataset = CVFGATGeometricDataset(
    device, dataset="complete_graph_n5", program="graph_coloring"
)  # list of Data objects
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=1):
        super().__init__()
        # First GAT layer
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads)
        # Second GAT layer
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        # x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        index = (
            torch.LongTensor([[i] * dataset[0].num_nodes for i in range(batch_size)])
            .to(device)
            .flatten()
        )
        # print("index", index, "x", x)
        return global_mean_pool(x, index).to(device)


# Model, optimizer, loss
model = GAT(
    in_channels=dataset.num_node_features,
    hidden_channels=8,
    out_channels=1,
    heads=8,
)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

In [5]:
# Training loop
def train():
    model.train()
    optimizer.zero_grad()
    total_loss = torch.FloatTensor([0.0]).to(device=device)
    for batch in loader:
        out = model(batch.x, batch.edge_index)
        loss = F.mse_loss(out.flatten(), batch.y)
        total_loss += loss
        loss.backward()
        optimizer.step()
    return total_loss.item()


# # Testing
# def test():
#     model.eval()
#     out = model(data.x, data.edge_index)
#     pred = out.argmax(dim=1)
#     accs = []
#     for mask in [data.train_mask, data.val_mask, data.test_mask]:
#         correct = (pred[mask] == data.y[mask]).sum()
#         acc = int(correct) / int(mask.sum())
#         accs.append(acc)
#     return accs


for epoch in range(1, 101):
    loss = train()
    # train_acc, val_acc, test_acc = test()
    if epoch % 5 == 0:
        # print(
        #     f"Epoch {epoch:03d}, Loss: {loss:.4f}, Train: {train_acc:.4f}, Val: {val_acc:.4f}, Test: {test_acc:.4f}"
        # )
        print(f"Epoch {epoch:03d}, Loss: {loss:.4f}")

index tensor([ 0,  0,  0,  0,  0,  1,  1,  1,  1,  1,  2,  2,  2,  2,  2,  3,  3,  3,
         3,  3,  4,  4,  4,  4,  4,  5,  5,  5,  5,  5,  6,  6,  6,  6,  6,  7,
         7,  7,  7,  7,  8,  8,  8,  8,  8,  9,  9,  9,  9,  9, 10, 10, 10, 10,
        10, 11, 11, 11, 11, 11, 12, 12, 12, 12, 12, 13, 13, 13, 13, 13, 14, 14,
        14, 14, 14, 15, 15, 15, 15, 15], device='cuda:0') x tensor([[0.9036],
        [0.9036],
        [0.9036],
        [0.9036],
        [0.9036],
        [0.7611],
        [0.7611],
        [0.7611],
        [0.7611],
        [0.7611],
        [0.8212],
        [0.8212],
        [0.8212],
        [0.8212],
        [0.8212],
        [0.5592],
        [0.5592],
        [0.5592],
        [0.5592],
        [0.5592],
        [0.3846],
        [0.3846],
        [0.3846],
        [0.3846],
        [0.3846],
        [0.8212],
        [0.8212],
        [0.8212],
        [0.8212],
        [0.8212],
        [0.4676],
        [0.4676],
        [0.4676],
        [0.4676],
  

RuntimeError: Expected index [80] to be no larger than self [16] apart from dimension 0 and to be no larger size than src [25]